In [1]:
import sys
import os
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "..")))

In [2]:
import dataclasses
import logging
import os
import sys
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple

import evaluate
import simple_parsing
import wandb
import torch
from dotenv import load_dotenv
import transformers
from peft import LoraConfig, TaskType, get_peft_model
from transformers import (
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
    WhisperForConditionalGeneration,
    WhisperProcessor,
)

In [3]:
from data import datasets, registry, types, partitioning

In [4]:
@dataclasses.dataclass
class LoraConfigArgs:
    r: int = 8
    lora_alpha: int = 32
    lora_dropout: float = 0.1
    target_modules: Optional[List[str]] = None



In [5]:
@dataclasses.dataclass
class TrainConfig(simple_parsing.helpers.Serializable):
    # Model parameters
    model_id: str = "openai/whisper-tiny"
    language: str = "en"
    task: str = "transcribe"
    # List of strings to match against parameter names. If None, all parameters are trainable.
    target_modules: Optional[List[str]] = None
    
    # LoRA Configuration
    lora_config: Optional[LoraConfigArgs] = None

    # Data parameters
    train_sets: List[Dict[str, Any]] = simple_parsing.list_field()
    val_sets: List[Dict[str, Any]] = simple_parsing.list_field()
    eval_sets: List[Dict[str, Any]] = simple_parsing.list_field()

    train_dataset_args: types.TrainDatasetArgs = simple_parsing.field(
        default_factory=types.TrainDatasetArgs
    )
    val_dataset_args: types.ValDatasetArgs = simple_parsing.field(
        default_factory=types.ValDatasetArgs
    )
    eval_dataset_args: types.EvalDatasetArgs = simple_parsing.field(
        default_factory=types.EvalDatasetArgs
    )

    # Scaling parameters
    # List of fractions to train on, e.g. [0.1, 0.5, 1.0]
    data_fractions: List[float] = simple_parsing.list_field(1.0)
    # Number of random subsets to train for each fraction
    num_subsets: int = 1

    # Training parameters
    output_dir: Path = Path("outputs")
    num_epochs: float = 3.0
    batch_size: int = 4
    grad_accum_steps: int = 1
    learning_rate: float = 1e-5
    warmup_steps: int = 500
    max_steps: int = 0  # if > 0, overrides num_epochs
    fp16: bool = True
    seed: int = 42

    # Evaluation
    do_eval: bool = True
    eval_steps: int = 1000

    def get_train_sets(self) -> List[types.DatasetConfig]:
        return [types.DatasetConfig.from_dict(ds) for ds in self.train_sets]

    def get_val_sets(self) -> List[types.DatasetConfig]:
        return [types.DatasetConfig.from_dict(ds) for ds in self.val_sets]


In [6]:
def prepare_dataset(
    data_opts: List[types.DatasetConfig],
    data_args: types.VoiceDatasetArgs,
    max_samples: Optional[int] = None,
) -> datasets.SizedIterableDataset:
    data_sets = []
    for ds_config in data_opts:
        # Register if not already registered (or just ensure it's in the map)
        if ds_config.name not in registry.DATASET_MAP:
             registry.register_datasets([ds_config])
        
        ds = registry.create_dataset(ds_config.name, data_args, verbose=True)
        data_sets.append(ds)

    # Interleave if multiple datasets
    if len(data_sets) > 1:
        dataset = datasets.InterleaveDataset(data_sets)
    else:
        dataset = data_sets[0]

    # Apply range limit if specified
    if max_samples is not None:
        dataset = datasets.Range(dataset, max_samples)
    elif data_args.max_samples != -1:
        dataset = datasets.Range(dataset, data_args.max_samples)
        
    return dataset


In [7]:
def set_trainable_parameters(model: torch.nn.Module, target_modules: Optional[List[str]]) -> None:
    """
    Sets the trainability of the model parameters based on the target_modules list.
    If target_modules is None or empty, all parameters are trainable.
    Otherwise, only parameters whose names contain one of the target_modules strings are trainable.
    """
    if not target_modules:
        return

    trainable_params = 0
    all_params = 0
    
    for name, param in model.named_parameters():
        all_params += param.numel()
        if any(target in name for target in target_modules):
            param.requires_grad = True
            trainable_params += param.numel()
        else:
            param.requires_grad = False
            
    logging.info(f"Set trainability based on target_modules: {target_modules}")
    logging.info(f"Trainable params: {trainable_params} / {all_params} ({trainable_params/all_params:.2%})")


In [8]:
def run_inference(
    model: torch.nn.Module,
    processor: WhisperProcessor,
    dataset: datasets.SizedIterableDataset,
    device: torch.device,
) -> Tuple[Dict[str, float], List[str], List[str]]:
    """
    Runs inference on the dataset and computes WER.
    Returns metrics, predictions, and references.
    """
    logging.info("Starting inference...")
    model.eval()
    metric = evaluate.load("wer")
    
    predictions = []
    references = []
    
    # Iterate over dataset
    # Note: This assumes dataset yields samples with 'audio' and 'text'
    # We need to process them manually since we are not using the Trainer's loop
    
    from tqdm import tqdm
    
    for i, sample in tqdm(enumerate(dataset), desc="Inference"):
        # Process audio
        audio = sample.audio
        input_features = processor(
            audio, sampling_rate=16000, return_tensors="pt"
        ).input_features
        input_features = input_features.to(device)
        
        # Generate
        with torch.no_grad():
            generated_ids = model.generate(input_features)
        
        # Decode
        transcription = processor.batch_decode(generated_ids, skip_special_tokens=True)[0]
        reference = sample.text
        
        # Normalization (simple lowercasing for now)
        transcription = transcription.lower()
        reference = reference.lower()
        
        predictions.append(transcription)
        references.append(reference)
        
        if i < 3:
            logging.info(f"Sample {i}:")
            logging.info(f"  Ref: {reference}")
            logging.info(f"  Pred: {transcription}")

    wer = metric.compute(predictions=predictions, references=references)
    logging.info(f"Final Inference WER: {wer}")
    
    return {"wer": wer}, predictions, references

In [9]:
class DataCollatorSpeechSeq2SeqWithPadding:
    def __init__(self, processor):
        self.processor = processor

    def __call__(
        self, features: List[Dict[str, Any]]
    ) -> Dict[str, torch.Tensor]:
        # split inputs and labels since they have to be of different lengths and need different padding methods
        # first treat the audio inputs by simply returning torch tensors
        input_features = [
            {"input_features": feature["audio"]["array"]} for feature in features
        ]
        batch = self.processor.feature_extractor.pad(
            input_features, return_tensors="pt"
        )

        # get the tokenized label sequences
        label_features = [{"input_ids": feature["text_input_ids"]} for feature in features]
        # pad the labels to max length
        labels_batch = self.processor.tokenizer.pad(
            label_features, return_tensors="pt"
        )

        # replace padding with -100 to ignore loss correctly
        labels = labels_batch["input_ids"].masked_fill(
            labels_batch.attention_mask.ne(1), -100
        )

        # if bos token is appended in previous tokenization step,
        # cut bos token here as it's append later anyways
        if (
            labels[:, 0] == self.processor.tokenizer.bos_token_id
        ).all().cpu().item():
            labels = labels[:, 1:]

        batch["labels"] = labels
        return batch

In [10]:
load_dotenv()
logging.basicConfig(level=logging.INFO)

In [11]:
# Register pre-defined datasets
import data.configs
registry.register_datasets(data.configs.ALL_CONFIGS)

In [13]:
config = TrainConfig.load("../exps/001_s_1.yaml")

In [14]:
transformers.set_seed(config.seed)

In [15]:
processor = WhisperProcessor.from_pretrained(
    config.model_id, language=config.language, task=config.task
)

In [16]:
# Prepare validation dataset (fixed across experiments)
val_dataset = prepare_dataset(
    config.get_val_sets(), config.val_dataset_args
)

INFO:root:Creating dataset spgispeech_2 with config:
{
  "name": "spgispeech_2",
  "base": null,
  "path": "/mnt/asr-data-scaling/data/local_datasets/spgispeech_2",
  "subset": null,
  "splits": [
    {
      "name": "train",
      "num_samples": 156753,
      "split": "TRAIN"
    },
    {
      "name": "validation",
      "num_samples": 6195,
      "split": "VALIDATION"
    },
    {
      "name": "test",
      "num_samples": 6932,
      "split": "TEST"
    }
  ],
  "transcript_field": "transcript",
  "audio_field": "audio",
  "audio_path_field": null,
  "audio_root": null,
  "streaming": false,
  "features": null,
  "eval_config": null
}


Loading dataset shards:   0%|          | 0/23 [00:00<?, ?it/s]

In [22]:
iter_val = iter(val_dataset)

In [25]:
sample_1 = next(iter_val)
sample_2 = next(iter_val)
sample_3 = next(iter_val)


In [27]:
from IPython.display import display, Audio
import textwrap
sample = sample_3
print(textwrap.fill(sample.text, width=80))
Audio(sample.audio,rate=16000)

Thank you for your participation in the call today . I want to thank our
employees, our E & P partners and investors . | this concludes today's
conference call . You may Disconnect your lines at this time . Thank you for
your participation, and have a great day.


In [19]:
# Wrap validation dataset with processing
class WhisperDataproc(datasets.Dataproc):
    def _process(self, sample):
        # Process audio
        audio = sample.audio
        input_features = processor(
            audio, sampling_rate=16000, return_tensors="np"
        ).input_features[0]
        
        # Process text
        labels = processor(text=sample.text).input_ids
        
        return {
            "audio": {"array": input_features},
            "text_input_ids": labels,
        }

In [20]:
val_dataset_proc = WhisperDataproc(val_dataset)

In [21]:
full_train_dataset = prepare_dataset(
    config.get_train_sets(), config.train_dataset_args
)

INFO:root:Creating dataset spgispeech-s with config:
{
  "name": "spgispeech-s",
  "base": null,
  "path": "kensho/spgispeech",
  "subset": "S",
  "splits": [
    {
      "name": "train",
      "num_samples": 77073,
      "split": "TRAIN"
    },
    {
      "name": "validation",
      "num_samples": 39304,
      "split": "VALIDATION"
    },
    {
      "name": "test",
      "num_samples": 39341,
      "split": "TEST"
    }
  ],
  "transcript_field": "transcript",
  "audio_field": "audio",
  "audio_path_field": null,
  "audio_root": null,
  "streaming": true,
  "features": null,
  "eval_config": null
}


In [24]:
total_train_samples = len(full_train_dataset)
total_train_samples

77073

In [25]:
config.data_fractions

[1.0]